# Lab 1 · Search and optimization

**Format:** Python/Jupyter, 90 minutes. Runs locally or in Colab. The second search example also needs `L01_maze.txt`.
Requires NumPy, Matplotlib and IPython; no GPU or helper Python files. Keep the supplied maze text file beside the notebook; in Colab upload it through the Files panel.
Start with **Runtime/Kernel → Restart and run all**. Provided support code is collapsed in interfaces that support this.

**Learning objectives:** Formulate a grid-route problem, explain A* priorities, implement Manhattan distance, and compare hill climbing with simulated annealing.

**Time budget:** 10 min setup and definitions; 45 min Tasks 1–2B; 20 min Task 3; 15 min written answers.

Read the worked example, complete the small `NotImplementedError` task(s), rerun the public checks, then write short explanations.
An uncompleted notebook prints TODO messages and skips dependent experiments; it is not a completed submission.

## What you will do

This lab studies two different problems. In **path search**, find a sequence of legal moves from a start to a goal with minimum total cost. In **local optimization**, choose a candidate with a small objective value by repeatedly trying nearby candidates.

| Task | Your work | What to keep in the notebook |
|---|---|---|
| 1. Understand the route problem | Read the definitions and run the supplied uniform-cost search | Route plot, path cost and expansion count |
| 2. Guide the search | Implement `student_heuristic(state, goal)` and run the supplied A* comparison | Passing checks, second route plot and completed comparison table |
| 2B. Search a larger maze | Load `L01_maze.txt` and compare UCS with A* | Side-by-side explored regions and a second search table |
| 3. Compare local optimization methods | Run the supplied experiment for ten fixed seeds | Both plots and the printed mean/standard deviation for each method |
| 4. Interpret | Answer the four questions at the end | One or two sentences per question |

**Coding scope:** replace the `raise NotImplementedError(...)` line in `student_heuristic` with your own calculation. This is the only required function to implement in this lab. A*, the local-search algorithms, plotting and experiment loops are provided. Read their explanations below; expand the support cell if you want to inspect the implementation. Homework 1 asks you to implement BFS and an annealing probability separately.

Run the following two setup cells first. They import NumPy as `np` and Matplotlib as `plt`, define the fixed grid and supplied algorithms, and define `check_task`, which runs the public checks. A TODO message means your implementation is still missing; after editing the function, rerun its cell and the check/experiment cell below it.

In [ ]:
"""Small, inspectable search examples. Written for the 2026 FAI course."""
import heapq
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

GRID = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
])
START, GOAL = (0, 0), (4, 5)

def neighbours(grid, state):
    r, c = state
    for dr, dc in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1] and grid[nr, nc] == 0:
            yield (nr, nc)

def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

def astar(grid, start, goal, heuristic=manhattan):
    """Unit-cost grid search; improved paths may reopen a state."""
    counter = itertools.count()
    frontier = [(heuristic(start, goal), next(counter), 0, start)]
    cost, parent, expanded = {start: 0}, {start: None}, []
    while frontier:
        _, _, g, state = heapq.heappop(frontier)
        if g != cost[state]:
            continue  # An obsolete queue entry, superseded by a cheaper path.
        expanded.append(state)
        if state == goal:
            path = []
            while state is not None:
                path.append(state)
                state = parent[state]
            return path[::-1], g, expanded
        for nxt in neighbours(grid, state):
            candidate = g + 1
            if candidate < cost.get(nxt, float('inf')):
                cost[nxt], parent[nxt] = candidate, state
                heapq.heappush(frontier, (candidate + heuristic(nxt, goal), next(counter), candidate, nxt))
    return [], float('inf'), expanded

def draw_grid(grid, path=(), expanded=(), title='Search', start=START, goal=GOAL):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.imshow(grid, cmap='Greys', vmin=0, vmax=1)
    if expanded:
        ax.scatter([s[1] for s in expanded], [s[0] for s in expanded], c='#65b8a6', s=110, label='expanded')
    if path:
        ax.plot([s[1] for s in path], [s[0] for s in path], 'o-', color='#d15a39', label='path')
    for label, state in [('S', start), ('G', goal)]:
        ax.text(state[1], state[0], label, ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set(xticks=range(grid.shape[1]), yticks=range(grid.shape[0]), title=title, xlabel='column', ylabel='row')
    fig.tight_layout()
    return fig

def landscape(x):
    return 0.12 * np.asarray(x)**2 + np.sin(3 * np.asarray(x))

def local_search(anneal=False, seed=0, steps=200, temperature=0.8):
    rng = np.random.default_rng(seed)
    x, best = 2.5, 2.5
    history = [float(landscape(x))]
    for k in range(steps):
        candidate = float(np.clip(x + rng.normal(0, 0.4), -4, 4))
        delta = float(landscape(candidate) - landscape(x))
        T = max(temperature * 0.98**k, 1e-12)
        if delta <= 0 or (anneal and rng.random() < np.exp(-delta / T)):
            x = candidate
        if landscape(x) < landscape(best):
            best = x
        history.append(float(landscape(x)))
    return best, np.array(history)

In [ ]:
def check_task(label, check):
    try:
        check()
    except NotImplementedError:
        print(label + ': TODO — complete the marked function, then rerun.')
        return False
    except AssertionError as error:
        print(label + ': check failed — ' + str(error))
        return False
    print(label + ': public checks passed.')
    return True

## Task 1 · Model the route problem and run a baseline

A **state** is the agent's position `(row, column)`, with zero-based indices. An **action** moves one cell right, down, left or up. A move is legal only if its destination is inside the grid and open. A **path** lists all visited positions, including the start and goal. Every move costs 1, so a path containing `n` positions costs `n - 1`.

The supplied `GRID` has five rows and six columns: `0` means open and `1` means a wall. The initial state is `START = (0, 0)`; the goal test is equality with `GOAL = (4, 5)`. Rows increase downwards. The function `neighbours(grid, state)` yields legal next states in right/down/left/up order.

```text
        column: 0 1 2 3 4 5
row 0           S 0 0 0 0 0
row 1           0 1 1 0 1 0
row 2           0 0 0 0 1 0
row 3           1 1 0 1 0 0
row 4           0 0 0 0 0 G
```

Here `S` and `G` are open cells. The objective is to find a legal path of minimum total cost; several different paths may have the same minimum cost.

### How the supplied A* algorithm works

The **frontier** contains discovered candidates waiting to be processed. For a state `s`, `g(s)` is the cost of its best route found so far from the start, and `h(s)` estimates the remaining cost to the goal. A* processes the entry with smallest **priority** `f(s) = g(s) + h(s)`; ties are processed in insertion order.

1. Set the start's cost to 0 and add it to the frontier with priority `h(start)`.
2. Remove the frontier entry with lowest priority. Skip it if a cheaper route to that state has since been found.
3. If it is the goal, reconstruct the route by following stored parents backwards and then reverse that list.
4. Otherwise, consider each legal neighbour. Its proposed cost is `g(s) + 1`. If this improves its stored cost (or it has no cost yet), record the cost and parent and add a new frontier entry.
5. Repeat until the goal is selected or the frontier is empty. An empty frontier means no route exists.

**Uniform-cost search (UCS)** uses `h(s) = 0`, so it orders candidates only by their cost from the start. **Breadth-first search (BFS)** uses a first-in, first-out queue and explores paths by number of moves. Because every move costs 1 here, UCS and BFS find the same minimum path cost.

**Run the cell below without editing it.** `astar(...)` returns `(path, cost, expanded)`. Here `expanded` records processed, non-obsolete entries, including the goal; `len(expanded)` is the expansion count used in our comparison. If no route exists, the return values include `path = []` and `cost = infinity`. In the plot, black cells are walls, green markers show processed states, and the orange line is the route. Record the printed cost and count in the Task 2 table.

In [ ]:
ucs_path, ucs_cost, ucs_expanded = astar(GRID, START, GOAL, lambda s, g: 0)
print('UCS path cost:', ucs_cost, '| expanded nodes:', len(ucs_expanded))
draw_grid(GRID, ucs_path, ucs_expanded, 'Uniform-cost search')
plt.show()

## Task 2 · Implement the Manhattan heuristic

A heuristic is **admissible** if it never overestimates the minimum remaining path cost. For four-directional unit-cost movement, the **Manhattan distance** is

`h((r, c), (rg, cg)) = abs(r - rg) + abs(c - cg)`.

For example, from `(1, 2)` to `(4, 0)`, the row difference is 3 and the column difference is 2, giving an estimate of 5 moves. This ignores walls. A legal move changes just one coordinate by one unit; walls may force extra moves. At the goal, both coordinate differences are zero.

**Implement:** replace the TODO in `student_heuristic(state, goal)` below. Both arguments are `(row, column)` tuples of integers. Return the nonnegative integer Manhattan distance. Calculate it yourself with coordinate differences and `abs`; do not call the supplied `manhattan` function or a search algorithm. The function does not need the grid.

**Run and record:** run your function cell, then the next cell to check examples and run A* using your function. Leave the fixed grid and endpoints unchanged. Copy both algorithms' printed measurements into the table after the experiment. Equal path costs are expected; the expansion counts need not differ on every grid.

In [ ]:
def student_heuristic(state, goal):
    # Return the sum of the absolute row and column differences.
    raise NotImplementedError('Implement Manhattan distance here.')

In [ ]:
def check_heuristic():
    assert student_heuristic((1,2), (4,0)) == 5
    assert student_heuristic((2,3), (2,3)) == 0
    assert student_heuristic((4,0), (1,2)) == 5
ready = check_task('Heuristic', check_heuristic)
if ready:
    astar_path, astar_cost, astar_expanded = astar(GRID, START, GOAL, student_heuristic)
    print('A* path cost:', astar_cost, '| expanded nodes:', len(astar_expanded))
    draw_grid(GRID, astar_path, astar_expanded, 'A* with Manhattan distance')
    plt.show()

**Your measurements:** replace each ellipsis with a printed value.

| Algorithm | Path cost (number of moves) | Expanded entries |
|---|---:|---:|
| UCS (`h = 0`) | … | … |
| A* (Manhattan distance) | … | … |

## Task 2B · A larger maze loaded from a text file

The small grid has few distractions: UCS processes 22 entries and Manhattan A* processes 20. This second example is an irregular maze of single-cell-wide corridors, winding branches, junctions and 104 dead ends. All open cells are connected, but many branches lead away from the goal. The layout is fixed in the text file so everyone compares the same problem.

**Input file:** `L01_maze.txt`, supplied alongside this notebook. Each text line is one grid row, all rows have equal length, and each character represents one cell:

| Character | Meaning |
|---|---|
| `#` | Wall |
| `.` | Open cell |
| `S` | Start (exactly one, also open) |
| `G` | Goal (exactly one, also open) |

The provided loader reads the entire structure and both endpoints from the file; no maze is constructed in the notebook. It converts walls to 1 and other cells to 0, so the same search functions apply. Coordinates are zero-based `(row, column)`, counting every character including the outer walls. The supplied maze has 31 rows and 51 columns, `S = (15, 1)` and `G = (15, 49)`.

**Local Jupyter:** keep `L01_maze.txt` beside the notebook, or leave it in the course's `labs` folder. The loader searches these locations from the working directory and its parents, which also supports the instructor solution folder. **Colab:** upload the notebook, then use the left-hand Files panel → Upload to upload `L01_maze.txt` into the runtime before running this cell. If you move the notebook elsewhere, take the text file with it.

**Your task:** run the loader and comparison cells after completing `student_heuristic`. No additional function implementation is required. Retain the side-by-side figure, fill the maze comparison table, and discuss the difference in Question 1. Keep the supplied maze unchanged for the required measurements; edit a copy for further experiments.

In [ ]:
from pathlib import Path


def load_maze(path):
    rows = Path(path).read_text(encoding='utf-8').splitlines()
    if not rows or not rows[0] or any(len(row) != len(rows[0]) for row in rows):
        raise ValueError('Maze must be nonempty with equal-length rows.')
    if set(''.join(rows)) - set('#.SG'):
        raise ValueError('Use only #, ., S and G; do not include spaces or blank lines.')
    starts = [(r, c) for r, row in enumerate(rows) for c, value in enumerate(row) if value == 'S']
    goals = [(r, c) for r, row in enumerate(rows) for c, value in enumerate(row) if value == 'G']
    if len(starts) != 1 or len(goals) != 1:
        raise ValueError('Maze must contain exactly one S and one G.')
    grid = np.array([[int(value == '#') for value in row] for row in rows], dtype=int)
    return grid, starts[0], goals[0]


maze_file = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for relative in ['L01_maze.txt', 'labs/L01_maze.txt', 'New_Course_2026/labs/L01_maze.txt']:
        candidate = folder / relative
        if candidate.is_file():
            maze_file = candidate
            break
    if maze_file is not None:
        break
if maze_file is None:
    raise FileNotFoundError(
        'L01_maze.txt is missing. Put it beside the notebook, or upload it '
        'using the Colab Files panel, then rerun this cell.'
    )

maze_grid, maze_start, maze_goal = load_maze(maze_file)
print('Loaded:', maze_file.name, '| shape:', maze_grid.shape)
print('Start:', maze_start, '| goal:', maze_goal, '| open cells:', int((maze_grid == 0).sum()))

In [ ]:
if ready:
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    maze_results = {}
    for label, heuristic in [('UCS (h = 0)', lambda s, g: 0),
                             ('A* (Manhattan)', student_heuristic)]:
        maze_results[label] = astar(maze_grid, maze_start, maze_goal, heuristic)

    print('Algorithm          | path cost | expanded entries')
    for label, (route, cost, expanded) in maze_results.items():
        print(f'{label:18} | {cost:9} | {len(expanded):16}')
    maze_ucs = maze_results['UCS (h = 0)']
    maze_astar = maze_results['A* (Manhattan)']
    assert maze_ucs[1] == maze_astar[1], 'Both methods should find the same optimal cost.'
    saved = len(maze_ucs[2]) - len(maze_astar[2])
    print(f'A* processed {saved} fewer entries ({100 * saved / len(maze_ucs[2]):.1f}% reduction).')
    print('Manhattan estimate at start:', student_heuristic(maze_start, maze_goal))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), sharex=True, sharey=True)
    colours = ListedColormap(['white', '#273444', '#8acbd6'])
    for ax, (label, (route, cost, expanded)) in zip(axes, maze_results.items()):
        display_grid = maze_grid.copy()
        for state in expanded:
            display_grid[state] = 2
        ax.imshow(display_grid, cmap=colours, vmin=0, vmax=2, interpolation='nearest')
        if route:
            ax.plot([s[1] for s in route], [s[0] for s in route], color='#dc681d', linewidth=2)
        for name, state in [('S', maze_start), ('G', maze_goal)]:
            ax.text(state[1], state[0], name, ha='center', va='center', weight='bold',
                    bbox=dict(facecolor='white', edgecolor='none', pad=1))
        ax.set(title=f'{label}: cost {cost}, expanded {len(expanded)}',
               xlabel='column', ylabel='row', xticks=range(0, maze_grid.shape[1], 5),
               yticks=range(0, maze_grid.shape[0], 5))
    fig.legend(handles=[Patch(facecolor='#273444', label='wall'),
                        Patch(facecolor='white', edgecolor='#777777', label='open, not expanded'),
                        Patch(facecolor='#8acbd6', label='expanded'),
                        Line2D([0], [0], color='#dc681d', linewidth=2, label='returned route')],
               loc='lower center', ncol=4)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    plt.show()
else:
    print('Maze comparison: TODO — complete student_heuristic and rerun its checks first.')

**Maze measurements:** copy the results from the comparison above.

| Algorithm | Path cost | Expanded entries |
|---|---:|---:|
| Maze UCS (`h = 0`) | … | … |
| Maze A* (Manhattan distance) | … | … |

**Why the heuristic helps here.** UCS prioritizes only the cost already spent, `g`. It spreads into all cheap-to-reach side passages before selecting a more distant goal. A* adds a lower bound on the cost still needed, `h`, and therefore postpones many states whose complete routes already look too expensive. Both searches stop when the goal is selected; UCS does not necessarily visit every reachable cell.

For a concrete example, the shortest route from `S = (15, 1)` to `(1, 1)` takes 26 moves through winding corridors. Its best `g` is therefore 26 and its Manhattan distance to `G = (15, 49)` is `14 + 48 = 62`, so its A* priority is `f = 88`. UCS processes this state before reaching the goal because `g = 26`. A* reaches the goal at cost 64 before it needs to process an entry with priority 88. The blue regions in the two panels make this avoided work visible.

Manhattan distance ignores the corridor walls: at the start it estimates 48, whereas the shortest legal route costs 64 because of the detours. A* still explores some unhelpful branches. It keeps the same optimal cost because Manhattan distance is a valid lower bound for these moves, and this implementation updates a state when it finds a cheaper route. The benefit is less search work, not a shorter route than UCS can find.

The measured expansion count is a work indicator, not a timing benchmark. Computing a heuristic also costs time, and its benefit depends on the map and how informative it is. On the first grid, many states have similar `g + h` values, leaving little work to avoid; the large maze provides many clearly unpromising branches.

## Task 3 · Run and understand local optimization

Now a **candidate** is a single real number `x` in `[-4, 4]`. The **objective function** is `f(x) = 0.12*x**2 + sin(3*x)` (sine in radians). We want to minimize it: a lower value is better. There is no designated goal cell or route. A **local minimum** is at least as good as nearby candidates; a **global minimum** is at least as good as every candidate in the allowed interval.

The supplied `local_search` starts at `x = 2.5` and makes 200 proposals. Each proposal adds a random normal perturbation with mean 0 and standard deviation 0.4, then clips the result to `[-4, 4]`. Define `delta = f(candidate) - f(current)`: negative means better, zero means equally good, and positive means worse.

| Method | If `delta <= 0` | If `delta > 0` |
|---|---|---|
| Hill climbing (minimization version) | Accept the candidate | Reject it |
| Simulated annealing | Accept the candidate | Accept with probability `exp(-delta / T)` |

Despite the name, this hill-climbing variant moves towards lower cost. Accepting means replacing the current candidate; rejecting means keeping it. Annealing accepts a worse proposal when a uniform random number in `[0, 1)` is below the stated probability. Its positive **temperature** decreases as `T = max(0.8 * 0.98**k, 1e-12)` for proposal indices `k = 0, ..., 199`. A higher temperature makes worse moves more likely, which can help leave a local minimum.

Both algorithms separately store the **best-so-far candidate**: it changes only when a lower cost is found. `local_search(...)` returns this best candidate and `history`, an array of the initial and subsequent **current** costs. `np.minimum.accumulate(history)` converts current costs to best-so-far costs for the plot.

**Run the next cell without editing it.** It compares both methods using seeds 0 through 9. A seed makes an individual run repeatable. Each method has the same starting point, proposal distribution and 200-proposal budget; their random draws and trajectories need not be identical. The left plot shows the objective; the right plot shows best-so-far costs for seed 0. The printout summarizes the final best costs across ten runs: mean and standard deviation (spread).

**Record:** retain both plots and copy the mean and standard deviation for each method into the table below. Smaller mean best cost is better. Use these measured results in Question 4; ten runs of one objective do not establish a universal winner.

In [ ]:
x = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x, landscape(x)); axes[0].set(title='Objective landscape', xlabel='x', ylabel='cost')
for anneal, label in [(False, 'hill climbing'), (True, 'annealing')]:
    final = []
    for seed in range(10):
        best, history = local_search(anneal=anneal, seed=seed)
        final.append(float(landscape(best)))
        if seed == 0:
            axes[1].plot(np.minimum.accumulate(history), label=label)
    print(label, 'mean best cost:', round(np.mean(final), 3), 'std:', round(np.std(final), 3))
axes[1].set(title='Best cost so far: seed 0', xlabel='proposal', ylabel='cost'); axes[1].legend()
fig.tight_layout(); plt.show()

**Your measurements:**

| Method | Mean best cost (10 seeds) | Standard deviation |
|---|---:|---:|
| Hill climbing | … | … |
| Simulated annealing | … | … |

## Task 4 · Explain your results

Write one or two sentences for each question. Use the outputs of this notebook.

1. Compare UCS and A* on both grids: give the costs and expansion counts, and the percentage reduction in expansions on the large maze. Why is the reduction larger there? Use the `g`, `h` and `f` values at `(1, 1)` to explain the avoided work. You may use three sentences for this answer.

   **Answer:** …

2. Why is Manhattan distance an admissible heuristic on this grid? Explain why walls cannot make the shortest legal route shorter than the estimate, and state its value at the goal.

   **Answer:** …

3. Suppose annealing accepts a candidate with cost 0.7 when the current cost is 0.2 and the best recorded cost is 0.1. What are the new current and best costs? Explain why current cost can increase while best-so-far cost cannot.

   **Answer:** …

4. Which method had the lower mean best cost in your ten-run experiment? Report both means and standard deviations. Does simulated annealing guarantee the global minimum within these 200 proposals? Explain.

   **Answer:** …

**Completion checklist:** your heuristic passes the checks; all three measurement tables are filled; the small-grid route figures, the maze comparison figure, the local-optimization figure and experiment printouts are present; all four answers are written. Restart the kernel and run all cells before handing in the notebook if your instructor requests a lab submission.

**Optional extension (not required):** copy `GRID`, block every open neighbour of `GOAL` while keeping the goal itself open, and call `astar` on that copy. Check that it returns an empty path and infinite cost. Keep the required experiments above unchanged.

**Sources:** supplied `Curs-02.pdf`, `FAI_Lecture6-2025.pdf`, and `FAI2025-lab4.ipynb`. The examples and implementations here are newly written; algorithmic themes are adapted from those materials.